In [8]:
import numpy as np
import pandas as pd
import optuna
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from catboost import CatBoostClassifier

# 0. Загрузка датасетов из CSV
train = pd.read_csv('data/train.csv', index_col='id')
test = pd.read_csv('data/test.csv', index_col='id')

# 1. Кастомный трансформер для генерации признаков без утечки данных
class BankFeatureGenerator(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.bins = [17, 38, 42, 45, 48, 72]
        self.labels = ['18-38', '39-42', '43-45', '46-48', '49-72']
        self.age_salary_map_ = {}
        self.country_bal_map_ = {}
        self.global_median_salary_ = 0.0
        self.global_mean_balance_ = 0.0

    def fit(self, X, y=None):
        df = X.copy()
        df['AgeGroup'] = pd.cut(df['Age'], bins=self.bins, labels=self.labels).astype(str)
        
        self.age_salary_map_ = df.groupby('AgeGroup')['EstimatedSalary'].median().to_dict()
        self.country_bal_map_ = df.groupby('Geography')['Balance'].mean().to_dict()
        self.global_median_salary_ = df['EstimatedSalary'].median()
        self.global_mean_balance_ = df['Balance'].mean()
        return self

    def transform(self, X):
        df = X.copy()
        df['AgeGroup'] = pd.cut(df['Age'], bins=self.bins, labels=self.labels).astype(str)
        
        df['WealthIndex'] = df['Balance'] * df['CreditScore'] / 10000
        df['ZeroBalance'] = (df['Balance'] == 0).astype(int)
        df['SalaryPerYear'] = df['EstimatedSalary'] / df['Age']
        
        median_sal = df['AgeGroup'].map(self.age_salary_map_).fillna(self.global_median_salary_)
        df['EstimatedSalaryDiff'] = median_sal / df['EstimatedSalary']
        
        df['AvgCountryBalance'] = df['Geography'].map(self.country_bal_map_).fillna(self.global_mean_balance_)
        df['AgeTenureRatio'] = df['Age'] / (df['Tenure'] + 1)
        df['HasCardAndActive'] = ((df['HasCrCard'] == 1) & (df['IsActiveMember'] == 1)).astype(int)
        df['LogEstimatedSalary'] = np.log1p(df['EstimatedSalary'])
        df['AgeSquared'] = df['Age'] ** 2
        
        conditions = [
            (df['NumOfProducts'] == 1),
            (df['NumOfProducts'] == 2),
            (df['NumOfProducts'] > 2)
        ]
        values = ['One Product', 'Two Products', 'More Than 2 Products']
        df['QuantityProducts'] = np.select(conditions, values, default='Other')
        
        return df.drop(columns=['NumOfProducts', 'EstimatedSalary', 'Age'])

# 2. Настройка препроцессора столбцов
cat_features_onehot = ['Geography', 'AgeGroup', 'QuantityProducts']
cat_features_ordinal = ['Gender', 'HasCrCard', 'IsActiveMember', 'ZeroBalance', 'HasCardAndActive']
num_features = [
    'CreditScore', 'Tenure', 'Balance', 'WealthIndex', 'SalaryPerYear',
    'EstimatedSalaryDiff', 'AvgCountryBalance', 'AgeTenureRatio',
    'LogEstimatedSalary', 'AgeSquared'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features_onehot),
        ('cat_bin', OrdinalEncoder(), cat_features_ordinal)
    ],
    remainder='drop'
)

# 3. Разделение сырых данных на обучающую и отложенную валидационную выборки
X_raw = train.drop(columns=['CustomerId', 'Surname', 'Exited'])
y = train['Exited'].values

X_train, X_val, y_train, y_val = train_test_split(
    X_raw, y, test_size=0.3, random_state=42, stratify=y
)

# 4. Функция оптимизации Optuna с полным пайплайном внутри кросс-валидации
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int('depth', 3, 7),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 15.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-5, 10.0, log=True),
        'loss_function': 'Logloss',
        'verbose': False,
        'random_state': 42
    }
    
    pipeline = Pipeline([
        ('features', BankFeatureGenerator()),
        ('prep', preprocessor),
        ('model', CatBoostClassifier(**params))
    ])
    
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# 5. Обучение лучшего пайплайна и проверка переобучения на X_val
best_pipeline = Pipeline([
    ('features', BankFeatureGenerator()),
    ('prep', preprocessor),
    ('model', CatBoostClassifier(**study.best_params, verbose=False, random_state=42))
])

best_pipeline.fit(X_train, y_train)

train_auc = roc_auc_score(y_train, best_pipeline.predict_proba(X_train)[:, 1])
val_auc = roc_auc_score(y_val, best_pipeline.predict_proba(X_val)[:, 1])

print(f"CV Best ROC-AUC: {study.best_value:.4f}")
print(f"Train ROC-AUC:   {train_auc:.4f}")
print(f"Val   ROC-AUC:   {val_auc:.4f}")
print(f"Переобучение:    {train_auc - val_auc:.4f}")

# 6. Финальное обучение на ВСЕХ доступных данных (X_raw, y) перед сабмитом
best_pipeline.fit(X_raw, y)

test_raw = test.drop(columns=['CustomerId', 'Surname'])
y_predict = best_pipeline.predict_proba(test_raw)[:, 1]

submission = pd.DataFrame({
    'id': test.index,
    'Exited': y_predict
})
submission.to_csv('submission.csv', index=False)


[I 2026-11-14 19:39:07,841] A new study created in memory with name: no-name-cb632910-7240-42aa-b08b-6bd080b86a9f
c:\Users\pashk\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
3 fits failed out of a total of 5.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\pashk\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\pashk\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\

CV Best ROC-AUC: 0.9326
Train ROC-AUC:   0.9385
Val   ROC-AUC:   0.9382
Переобучение:    0.0003
